# Assignment No. 2 — Data Cleaning and Preprocessing
**Course:** Data Science  
**Student Name:** Habibullah Azam  
**Dataset:** Students Performance in Exams  
**Dataset Source:** [Kaggle — Students Performance in Exams](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams)

## 1. Introduction
The purpose of this assignment is to inspect a real dataset, identify data-quality problems, clean the data, and prepare it for analysis or machine learning. The dataset contains students' demographic information and their math, reading, and writing scores.

**Objective:** Prepare a reliable dataset and demonstrate suitable preprocessing techniques. We will use **math score** as an example target for a demonstration model-ready dataset. This is a preprocessing assignment, not a claim that demographic information determines a student's ability.


## 2. Dataset Description
The Kaggle dataset is called **Students Performance in Exams**.

- **Expected size:** 1,000 rows and 8 columns.
- **Categorical columns:** `gender`, `race/ethnicity`, `parental level of education`, `lunch`, `test preparation course`.
- **Numerical columns:** `math score`, `reading score`, `writing score`.
- **Objective:** Explore student exam scores and prepare the data for further analysis.

### Load the CSV file
1. Download `StudentsPerformance.csv` from the Kaggle link above.
2. Run the code cell below and upload the CSV file when prompted.
3. If your downloaded file has a slightly different filename, select that file.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
uploaded = files.upload()

# Use the first uploaded CSV file
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)
display(df.head())


## 3. Initial Data Inspection
We inspect the shape, column names, data types, summary statistics, and missing values before changing anything. This helps us understand the dataset's condition.


In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types:")
display(df.dtypes.to_frame("Data type"))

print("\\nSummary of numerical columns:")
display(df.describe())

print("\\nMissing values before cleaning:")
display(df.isna().sum().to_frame("Missing count"))

print("\\nExact duplicate rows before cleaning:", df.duplicated().sum())


## 4. Standardize Column Names and Text
Column names are converted to lowercase and underscores so they are easier to use in Python. Text values are stripped of extra spaces and converted to lowercase to avoid categories such as `Male`, `male`, and ` male ` being treated as different values.


In [ ]:
# Make column names consistent
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace("/", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

# Clean spaces and letter case in text columns
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip().str.lower()

print("Standardized columns:", df.columns.tolist())
for col in text_cols:
    print(f"Unique values in {col}:", sorted(df[col].dropna().unique().tolist()))


## 5. Handle Missing Values
First, we measure missing values. If any categorical value is missing, we fill it with the most frequent category (mode). If any numerical value is missing, we fill it with the median. The median is less affected by extreme values than the mean.

We do not assume that missing values exist: the code reports the actual count first.


In [ ]:
print("Missing values before handling:")
display(df.isna().sum().to_frame("Missing count"))

for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode_values = df[col].mode(dropna=True)
            if len(mode_values) > 0:
                df[col] = df[col].fillna(mode_values.iloc[0])
            else:
                df[col] = df[col].fillna("unknown")

print("Missing values after handling:")
display(df.isna().sum().to_frame("Missing count"))


**Interpretation:** If the counts were already zero, no values needed to be imputed. If missing values were found, the code used the median for numerical columns and the mode for categorical columns. These are simple baseline methods; the best method depends on why data is missing.


## 6. Detect and Remove Duplicate Records
Duplicate rows can make some records count more than once. We count exact duplicate rows and remove them. We keep the first occurrence. Two students can legitimately have the same values, so only exact repeated rows are removed here.


In [ ]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().copy()
duplicates_after = df.duplicated().sum()

print("Duplicate rows found:", duplicates_before)
print("Duplicate rows remaining:", duplicates_after)
print("Shape after duplicate removal:", df.shape)


## 7. Correct Data Types and Validate Values
Exam scores should be numeric. We convert the three score columns to numeric values and check whether scores fall within the expected range of 0 to 100. Invalid entries become missing and are handled using the median. Values outside the score range are treated as invalid data rather than valid outliers.


In [ ]:
score_cols = ["math_score", "reading_score", "writing_score"]

for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Scores should be between 0 and 100; mark invalid values as missing
for col in score_cols:
    invalid_mask = ~df[col].between(0, 100) & df[col].notna()
    print(f"Invalid values in {col}:", int(invalid_mask.sum()))
    df.loc[invalid_mask, col] = np.nan
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print("\\nUpdated data types:")
display(df.dtypes.to_frame("Data type"))
print("Score ranges:")
display(df[score_cols].agg(["min", "max"]))


## 8. Identify and Handle Outliers
We use the Interquartile Range (IQR) rule to flag possible outliers in each score column. An outlier is not automatically an error. Since exam scores have a valid range of 0–100, a score such as 0 or 100 may be genuine. Therefore, we **report** IQR-flagged values but do not automatically delete or cap them. This avoids removing valid student results without evidence.


In [ ]:
def iqr_outlier_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower bound": lower,
        "Upper bound": upper,
        "Flagged values": int(mask.sum())
    })

display(pd.DataFrame({col: iqr_outlier_summary(df[col]) for col in score_cols}).T)

df[score_cols].plot(kind="box", figsize=(8, 4), title="Exam Score Boxplots")
plt.ylabel("Score")
plt.show()


**Cleaning decision:** IQR is used as a screening method. Because scores are bounded and unusual scores can be real, the flagged values are retained unless an actual data-entry error can be confirmed.


## 9. Identify Feature Types and Target
For this demonstration, `math_score` is the target variable. The remaining columns are candidate input features. In a real prediction task, the choice of target should match the research question.

- **Numerical:** `reading_score`, `writing_score`, and `math_score`.
- **Categorical nominal:** `gender`, `race_ethnicity`, `lunch`, and `test_preparation_course`.
- **Ordinal:** `parental_level_of_education` has a natural education-level order.
- **Binary:** `gender`, `lunch`, and `test_preparation_course` each have two categories in the expected dataset.
- **Date/time:** No date/time column is present in this dataset.


In [ ]:
target = "math_score"
features = [col for col in df.columns if col != target]

print("Target:", target)
print("Candidate features:", features)
print("\\nNumber of unique values per column:")
display(df.nunique().to_frame("Unique values"))

print("\\nCategory counts:")
for col in df.select_dtypes(include=["object", "string"]).columns:
    print(f"\\n{col}")
    print(df[col].value_counts(dropna=False))


## 10. Feature Engineering
We create `average_reading_writing_score` as the average of reading and writing scores. This can summarize performance across these two subjects. We do not use the target math score to create this feature, which avoids directly leaking the target into the engineered input feature.


In [ ]:
df["average_reading_writing_score"] = df[["reading_score", "writing_score"]].mean(axis=1)
display(df[["reading_score", "writing_score", "average_reading_writing_score", "math_score"]].head())


## 11. Encoding and Scaling Without Data Leakage
We split the data into training and testing sets **before** fitting preprocessing transformations. The imputer, encoder, and scaler are fitted only on the training set through a scikit-learn pipeline. The test set is transformed using those already-fitted steps.

- Nominal categories: one-hot encoding.
- Ordinal education level: ordinal encoding in a meaningful order.
- Numerical inputs: median imputation and standard scaling.
- Target: `math_score`; it is not scaled or included among the input features.

For a fair evaluation, we exclude `math_score` from the input features.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

target = "math_score"

# Do not include the target among input features
X = df.drop(columns=[target]).copy()
y = df[target].copy()

ordinal_col = ["parental_level_of_education"]
ordinal_order = [[
    "some high school",
    "high school",
    "some college",
    "associate's degree",
    "bachelor's degree",
    "master's degree"
]]

nominal_cols = [
    "gender", "race_ethnicity", "lunch", "test_preparation_course"
]
numeric_cols = [
    "reading_score", "writing_score", "average_reading_writing_score"
]

# Split first: preprocessing will be fitted only on the training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

nominal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

ordinal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(
        categories=ordinal_order,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_cols),
    ("nominal", nominal_pipeline, nominal_cols),
    ("ordinal", ordinal_pipeline, ordinal_col)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training rows:", X_train.shape[0])
print("Original testing rows:", X_test.shape[0])
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


## 12. Check Skewness and Transform if Required
We calculate skewness for numerical score features. A skewness value near zero suggests a more symmetrical distribution; a large positive or negative value suggests skewness. Score columns are bounded between 0 and 100, so we do not apply a transformation automatically. If a feature were strongly skewed and a transformation were justified, it should be fitted or selected using training data only. The score columns are not transformed here because this may make the results less interpretable.


In [ ]:
print("Skewness of numerical features:")
display(df[numeric_cols].skew().to_frame("Skewness"))

df[numeric_cols].hist(figsize=(10, 6), bins=20)
plt.suptitle("Distributions of Numerical Features")
plt.tight_layout()
plt.show()


## 13. Final Data Quality Checks
We check for remaining missing values, duplicate rows, and score ranges. We also save the cleaned dataset and the processed training/testing arrays as files.


In [ ]:
print("Final dataset shape:", df.shape)
print("\\nMissing values remaining:")
display(df.isna().sum().to_frame("Missing count"))

print("Exact duplicate rows remaining:", df.duplicated().sum())
print("\\nFinal score ranges:")
display(df[score_cols].agg(["min", "max"]))

df.to_csv("students_performance_cleaned.csv", index=False)
print("Saved: students_performance_cleaned.csv")


## 14. Conclusion
In this assignment, the Students Performance in Exams dataset was loaded and inspected. Column names and text categories were standardized; missing values, duplicate rows, data types, score ranges, and possible outliers were checked. A derived feature was created, and categorical and numerical inputs were encoded and scaled using a pipeline.

The train-test split was performed before fitting preprocessing steps. This helps prevent data leakage because transformations learn their settings from the training data only. Outlier flags were reviewed rather than removed automatically because valid exam scores may be unusual. The final cleaned dataset was saved as a CSV file.

**Important:** Run all cells in order, check the outputs, and make sure you can explain the purpose of each step in your own words before submission.
